# Per-Muscle and Overall Average Metrics — Sheffield Dataset

For each algorithm that has a `column_compare_*_sheffield.ipynb`, this notebook
reads the per-muscle result CSVs from `{algo}/codes/results_sheffield/`,
computes the mean of every metric across all samples, appends an `Overall_Mean`
row, and saves a summary CSV.  The final cell combines all `Overall_Mean` rows
into one comparison table with colour-coded Dice and Hausdorff columns.

In [7]:
import pathlib
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

In [8]:
# ── Paths ──────────────────────────────────────────────────────────────────────
EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results_sheffield'
SUMMARY_DIR.mkdir(exist_ok=True)

# ── Known Sheffield muscle names (used for column-prefix stripping) ─────────────
SHEFFIELD_MUSCLES = [
    'adductor_brevis', 'adductor_longus', 'adductor_magnus',
    'biceps_femoris_short', 'biceps_femoris_long', 'biceps_femoris',
    'gracilis', 'rectus_femoris', 'sartorius',
    'semimembranosus', 'semitendinosus',
    'vastus_intermedius', 'vastus_lateralis', 'vastus_medialis',
]

# Regex to strip '{muscle_name}_' prefix from column names
_SHEFF_PREFIX_RE = re.compile(
    r'^(' + '|'.join(re.escape(m) for m in sorted(SHEFFIELD_MUSCLES, key=len, reverse=True)) + r')_',
    re.I,
)

# ── Canonical metric patterns (specific before general) ────────────────────────
_METRIC_RE = [
    ('inter_slice_dice_pred', re.compile(r'inter_slice_dice_pred',        re.I)),
    ('inter_slice_dice_gt',   re.compile(r'inter_slice_dice_gt',          re.I)),
    ('dice',                  re.compile(r'^(?:lower_)?dice$',            re.I)),
    ('hausdorff',             re.compile(r'hausdorff',                    re.I)),
    ('jaccard',               re.compile(r'jaccard',                      re.I)),
    ('volume_similarity',     re.compile(r'volume_similarity',            re.I)),
    ('false_negative',        re.compile(r'false.?neg|falseNeg',          re.I)),
    ('false_positive',        re.compile(r'false.?pos|falsePo',           re.I)),
    ('bce',                   re.compile(r'\bbce\b|binary_cross_entropy', re.I)),
    ('boundary_iou_3d',       re.compile(r'boundary_iou',                 re.I)),
]


def canonical_metric(col: str):
    """Strip muscle-name prefix then match to a canonical metric name."""
    bare = _SHEFF_PREFIX_RE.sub('', col).rstrip(':')
    for name, pat in _METRIC_RE:
        if pat.search(bare):
            return name
    return None


def extract_muscle(stem: str):
    """Parse muscle name from df_{muscle}_{algo_tag}_sheffield filename stem."""
    s = stem.lower()
    if s.startswith('df_'):
        s = s[3:]
    if s.endswith('_sheffield'):
        s = s[:-10]
    for muscle in sorted(SHEFFIELD_MUSCLES, key=len, reverse=True):
        if s.startswith(muscle + '_') or s == muscle:
            return muscle
    return None


print('Helpers defined.')
print('Muscle prefix regex:', _SHEFF_PREFIX_RE.pattern[:80], '...')

Helpers defined.
Muscle prefix regex: ^(biceps_femoris_short|biceps_femoris_long|vastus_intermedius|vastus_lateralis|a ...


In [9]:
_FLOAT64_MAX = np.finfo(np.float64).max


def process_algorithm(label: str, results_dir: pathlib.Path):
    """Return a summary DataFrame for one algorithm, or None if no CSVs found."""
    csv_files = sorted(results_dir.glob('df_*.csv'))
    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []
    for csv_path in csv_files:
        # Skip per-algorithm summary files (summary_*.csv)
        muscle = extract_muscle(csv_path.stem)
        if muscle is None:
            print(f'  [skip] cannot identify muscle in {csv_path.name}')
            continue

        df = pd.read_csv(csv_path)
        metric_vals = {}
        for col in df.columns:
            metric_name = canonical_metric(col)
            if metric_name is None:
                continue
            s = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=np.float64)
            # Replace inf and extreme sentinel values with NaN
            s = np.where(np.isfinite(s) & (np.abs(s) < _FLOAT64_MAX), s, np.nan)
            if not np.isfinite(s).any():
                print(f'  [all non-finite] {csv_path.name} | {col}')
                continue
            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

        # Derived: inter-slice dice ratio (pred / gt)
        pred = metric_vals.get('inter_slice_dice_pred')
        gt   = metric_vals.get('inter_slice_dice_gt')
        if pred is not None and gt is not None and gt > 0:
            metric_vals['inter_slice_dice_ratio'] = pred / gt

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    numeric  = summary.select_dtypes(include='number').astype(np.float64)
    overall  = numeric.mean().rename('Overall_Mean')
    overall['algorithm'] = label
    summary  = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'
    return summary


print('process_algorithm defined.')

process_algorithm defined.


In [10]:
# ── Algorithm registry ─────────────────────────────────────────────────────────
# Each entry: (display_label, relative_path_to_results_sheffield_dir)
REGISTRY = [
    ('Dafne',                        'dafne/codes/results_sheffield'),
    ('MuscleMap Thigh',              'muscle_map_thigh/codes/results_sheffield'),
    ('MuscleMap WB',                 'muscle_map_wb/codes/results_sheffield'),
    ('Hirriririir',                  'multimodal-multiethnic/codes/results_sheffield'),
    ('MuSeg',                        'museg/codes/results_sheffield'),
    ('MedCLIP-SAMv2',                'medclipsamv2/codes/results_sheffield'),
    ('MedCLIP-SAMv2 Text+Boxes',     'medclipsamv2textboxes/codes/results_sheffield'),
    ('MedSegDiff',                   'medsegdiff/codes/results_sheffield'),
    ('SLM-SAM2',                     'muscle_map_wb+slmsam/codes/results_sheffield'),
]

print(f'{len(REGISTRY)} algorithms in registry:\n')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    if path.exists():
        n = len(list(path.glob('df_*.csv')))
        status = f'{n} CSVs'
    else:
        status = 'DIR MISSING'
    print(f'  {label:<35}  {status}')

9 algorithms in registry:

  Dafne                                12 CSVs
  MuscleMap Thigh                      13 CSVs
  MuscleMap WB                         13 CSVs
  Hirriririir                          11 CSVs
  MuSeg                                13 CSVs
  MedCLIP-SAMv2                        DIR MISSING
  MedCLIP-SAMv2 Text+Boxes             13 CSVs
  MedSegDiff                           DIR MISSING
  SLM-SAM2                             DIR MISSING


In [5]:
# ── Process all algorithms ─────────────────────────────────────────────────────
summaries = {}  # label -> DataFrame

for label, rdir in REGISTRY:
    results_dir = EVAL_DIR / rdir
    print(f'\n── {label} ──')
    df = process_algorithm(label, results_dir)
    if df is None:
        continue
    summaries[label] = df

    num_cols = df.select_dtypes(include='number').columns.tolist()
    display(
        df.reset_index()
        .style
        .format('{:.4f}', subset=num_cols)
        .hide(axis='index')
    )

    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_sheffield_avg_metrics.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')


── Dafne ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
adductor_longus,Dafne,0.004822,591.071739,0.002443,0.780416,0.992666,0.996336,0.028363,0.005992,0.502998,0.926139,0.543113
adductor_magnus,Dafne,0.072939,507.182891,0.038934,0.844616,0.917379,0.928577,0.113103,0.018186,0.514081,0.959396,0.535838
biceps_femoris_long,Dafne,0.088432,468.199364,0.047224,0.315036,0.712533,0.947241,0.093057,0.016768,0.699164,0.954679,0.732355
biceps_femoris_short,Dafne,0.032290,426.969420,0.016521,0.074822,0.486185,0.983182,0.195846,0.010382,0.769719,0.936863,0.821592
gracilis,Dafne,0.001189,473.642422,0.000596,0.125851,0.989937,0.999363,0.069311,0.001139,0.643151,0.904022,0.711433
rectus_femoris,Dafne,0.067734,567.910350,0.036149,0.457398,0.820231,0.955434,0.069052,0.019763,0.650347,0.950341,0.684330
sartorius,Dafne,0.015151,438.703781,0.007728,0.279116,0.939859,0.991165,0.063068,0.008849,0.652280,0.939520,0.694270
semimembranosus,Dafne,0.057864,409.282546,0.030307,0.610710,0.892293,0.958576,0.056276,0.027532,0.591381,0.955628,0.618840
semitendinosus,Dafne,0.081477,348.567831,0.044440,0.748595,0.886688,0.931143,0.032442,0.029356,0.470893,0.956392,0.492364
vastus_intermedius,Dafne,0.267995,450.360410,0.164441,0.732676,0.746872,0.671472,0.044219,0.104242,0.454063,0.960540,0.472717


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_sheffield\dafne_sheffield_avg_metrics.csv

── MuscleMap Thigh ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
adductor_brevis,MuscleMap Thigh,0.633934,36.691974,0.468493,0.058885,0.338490,0.000213,0.006303,0.272950,0.884957,0.907271,0.975405
adductor_longus,MuscleMap Thigh,0.709847,75.721816,0.554130,-0.345790,0.392613,0.000091,0.007651,0.263816,0.919490,0.926529,0.992403
adductor_magnus,MuscleMap Thigh,0.799174,188.270902,0.667976,-0.303634,0.302245,0.000206,0.026668,0.225283,0.947674,0.959187,0.987997
biceps_femoris_long,MuscleMap Thigh,0.761615,57.738961,0.618882,-0.400812,0.360024,0.000050,0.009598,0.245765,0.948124,0.954532,0.993287
biceps_femoris_short,MuscleMap Thigh,0.639997,39.922442,0.484584,-0.413931,0.440249,0.000092,0.006085,0.269492,0.944099,0.937400,1.007145
gracilis,MuscleMap Thigh,0.541894,40.544454,0.384807,-0.682177,0.579664,0.000039,0.005622,0.287696,0.930777,0.903922,1.029709
rectus_femoris,MuscleMap Thigh,0.685331,71.920183,0.542449,-0.302827,0.373719,0.000170,0.010251,0.264624,0.947706,0.950802,0.996743
sartorius,MuscleMap Thigh,0.581496,88.564309,0.421405,-0.240923,0.449797,0.000186,0.007956,0.290725,0.939793,0.940581,0.999162
semimembranosus,MuscleMap Thigh,0.691224,44.822388,0.530874,-0.518147,0.447104,0.000072,0.014549,0.159222,0.943028,0.955616,0.986827
semitendinosus,MuscleMap Thigh,0.691694,61.714675,0.534984,-0.490884,0.438375,0.000067,0.010919,0.210386,0.945389,0.956345,0.988544


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_sheffield\musclemap_thigh_sheffield_avg_metrics.csv

── MuscleMap WB ──


...

  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_sheffield\musclemap_wb_sheffield_avg_metrics.csv

── Hirriririir ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
adductor_magnus,Hirriririir,0.769524,402.193702,0.628130,-0.282572,0.320734,0.000371,0.031228,0.216827,0.952618,0.959187,0.993151
biceps_femoris_long,Hirriririir,0.630388,354.032759,0.467414,-0.691119,0.522539,0.000035,0.018947,0.182163,0.954350,0.954532,0.999809
biceps_femoris_short,Hirriririir,0.517804,326.266441,0.362031,-0.710037,0.585894,0.000088,0.011066,0.194162,0.942891,0.937400,1.005858
gracilis,Hirriririir,0.421039,347.376480,0.276792,-1.049334,0.709663,0.000023,0.011067,0.145501,0.940988,0.903922,1.041006
rectus_femoris,Hirriririir,0.649843,413.828338,0.497838,-0.505355,0.456034,0.000126,0.013791,0.211808,0.948144,0.950802,0.997204
sartorius,Hirriririir,0.483069,308.325949,0.328118,-0.665951,0.614495,0.000157,0.013525,0.171221,0.944369,0.940581,1.004027
semimembranosus,Hirriririir,0.611884,338.581528,0.448140,-0.643040,0.527741,0.000095,0.021332,0.176455,0.949520,0.955616,0.993621
semitendinosus,Hirriririir,0.619162,398.633747,0.455752,-0.689022,0.531771,0.000045,0.015942,0.167851,0.949392,0.956345,0.992729
vastus_intermedius,Hirriririir,0.540864,372.710899,0.373994,-0.805562,0.609146,0.000223,0.058414,0.162550,0.967523,0.960440,1.007375
vastus_lateralis,Hirriririir,0.604157,374.891381,0.439358,-0.654494,0.535128,0.000309,0.056031,0.156990,0.962965,0.954285,1.009096


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_sheffield\hirriririir_sheffield_avg_metrics.csv

── MuSeg ──


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
adductor_brevis,MuSeg,0.470956,41.710304,0.320322,0.679287,0.618906,0.305808,0.007086,0.237127,0.920782,0.907271,1.014892
adductor_longus,MuSeg,0.722862,26.007007,0.568086,0.815324,0.110850,0.384546,0.007496,0.291857,0.955045,0.926529,1.030777
adductor_magnus,MuSeg,0.763865,81.283907,0.634632,0.877341,0.136464,0.305206,0.030132,0.247656,0.969396,0.959187,1.010643
biceps_femoris_long,MuSeg,0.587525,130.412488,0.446547,0.694954,0.168161,0.525739,0.021452,0.197587,0.962920,0.954532,1.008787
biceps_femoris_short,MuSeg,0.552478,136.863858,0.406854,0.740019,0.273936,0.518106,0.007730,0.253876,0.958090,0.937400,1.022071
gracilis,MuSeg,0.459898,64.104024,0.312902,0.647056,0.259425,0.641771,0.006474,0.229199,0.956715,0.903922,1.058405
rectus_femoris,MuSeg,0.615595,63.759037,0.465705,0.689709,0.107665,0.507815,0.016348,0.217451,0.970988,0.950802,1.021230
sartorius,MuSeg,0.439512,131.281757,0.298045,0.780043,0.452891,0.598710,0.011273,0.215164,0.957190,0.940581,1.017658
semimembranosus,MuSeg,0.661770,74.631980,0.514784,0.749577,0.130691,0.460415,0.016350,0.203403,0.964826,0.955616,1.009637
semitendinosus,MuSeg,0.347868,177.751790,0.248308,0.565597,0.419887,0.745843,0.033721,0.095728,0.960125,0.956345,1.003953


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_sheffield\museg_sheffield_avg_metrics.csv

── MedCLIP-SAMv2 ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medclipsamv2\codes\results_sheffield

── MedCLIP-SAMv2 Text+Boxes ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medclipsamv2textboxes\codes\results_sheffield

── MedSegDiff ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medsegdiff\codes\results_sheffield

── SLM-SAM2 ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\muscle_map_wb+slmsam\codes\results_sheffield

Processed 5/9 algorithms.


In [6]:
# ── Combined Overall Means ─────────────────────────────────────────────────────
overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

if not overall_rows:
    print('No results yet — run the column_compare_*_sheffield notebooks first.')
else:
    combined = pd.concat(overall_rows)
    combined.index = [row['algorithm'] for _, row in combined.iterrows()]
    combined.index.name = 'algorithm'
    combined = combined.drop(columns='algorithm')
    combined = combined.drop(
        columns=['inter_slice_dice_pred', 'inter_slice_dice_gt'], errors='ignore'
    )

    # Prefer dice and hausdorff as the primary sort key
    if 'dice' in combined.columns:
        combined = combined.sort_values('dice', ascending=False)

    num_cols = combined.select_dtypes(include='number').columns.tolist()
    grad_cols = {col: 'RdYlGn' for col in ['dice', 'jaccard', 'boundary_iou_3d',
                                             'inter_slice_dice_ratio']
                 if col in num_cols}
    grad_cols_r = {col: 'RdYlGn_r' for col in ['hausdorff', 'false_negative',
                                                  'false_positive', 'bce']
                   if col in num_cols}

    def _bold_best(s: pd.Series, higher_is_better: bool) -> list[str]:
        """Bold the best (max or min) value in a column; ties all get bolded."""
        best = s.max() if higher_is_better else s.min()
        return ['font-weight: bold' if v == best else '' for v in s]

    styler = (
        combined.reset_index()
        .style
        .format('{:.4f}', subset=num_cols)
        .hide(axis='index')
    )
    for col, cmap in {**grad_cols, **grad_cols_r}.items():
        styler = styler.background_gradient(subset=[col], cmap=cmap, axis=0)
    for col in grad_cols:      # higher is better
        styler = styler.apply(_bold_best, subset=[col], higher_is_better=True)
    for col in grad_cols_r:    # lower is better
        styler = styler.apply(_bold_best, subset=[col], higher_is_better=False)

    display(styler)

    out_combined = SUMMARY_DIR / 'overall_means_sheffield.csv'
    combined.to_csv(out_combined, float_format='%.4f')
    print(f'Saved -> {out_combined}')

algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_ratio
MuscleMap WB,0.697428,30.420583,0.547437,-0.479335,0.421282,0.000089,0.014355,0.240754,0.994555
MuscleMap Thigh,0.687096,85.077213,0.535099,-0.376228,0.406130,0.000134,0.014151,0.242342,0.996846
Hirriririir,0.586737,364.037656,0.428907,-0.661066,0.539699,0.000160,0.026093,0.181098,1.003503
MuSeg,0.566311,95.750031,0.423090,0.724504,0.251823,0.506411,0.023029,0.212953,1.018194
Dafne,0.071711,466.056953,0.040343,0.553077,0.847710,0.934356,0.076330,0.024679,0.627932


Saved -> C:\Projects\dissector\eval_notebooks\summary_results_sheffield\overall_means_sheffield.csv
